## Importing Libraries

In [58]:
import os
from dotenv import load_dotenv

#langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings


In [59]:
load_dotenv()

True

In [60]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV Variables Loaded")

ENV Variables Loaded


### Loading the Data

In [61]:
DATA_FILE_PATH = os.path.join("data","hr_policy.txt")

### Data Ingestion

In [62]:
loader = TextLoader(DATA_FILE_PATH,encoding="utf-8")

documents = loader.load()

print("Data Loaded")
print("="*40)
print(documents)

Data Loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

### Langchain Documents
Langchain process eveything in form of documents

Page content -- actual data
Meta data -- extra info about data

In [63]:
len(documents)

1

In [64]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [65]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [66]:
print(f"total number of characters in the document: {len(documents[0].page_content)}")

total number of characters in the document: 2598


### Splitting the data

In [67]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [68]:
len(chunks)

9

In [69]:
print(chunks[-1].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


### Embed the data

In [70]:
embeddings_model = JinaEmbeddings(
    model_name="jina-embeddings-v2-base-en"
    )

### Store data in vector db

In [71]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings_model)

print("Chunks are stored", vector_store.index.ntotal)

Chunks are stored 9


In [72]:
test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query,k=2)

print(f"Query: {test_query}\n")

for i,match in enumerate(top_matches,start=1):
    print(f"Match {i}:")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

Match 1:
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

Match 2:
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### Tool

In [73]:
retriever = vector_store.as_retriever(search_kwargs={
    "k":3
})

def search_hr_policy(query:str)->str:
    """
    Search the HR policy document for information about leave, work from home, probation, notice period,
    reimbursement, code of conduct, holidays, or exit process.
    """
    matching_chunks = retriever.invoke(query)
    return "\n\n".join([chunk.page_content for chunk in matching_chunks])
    

### Data Retrieval

##### LLM

In [74]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=groq_key
)

llm.model_name

'openai/gpt-oss-120b'

In [75]:
test_response = llm.invoke("testing the llm")

In [76]:
test_response.content

'All systems are up and running! How can I assist you today?'

### AI AGENT

In [77]:
from langchain.agents import create_agent
hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
        You are a friendly HR assistant working for Acme Crop.
        Always use the search_hr_policy tool to look up
        facts before answering.
        If the answer isn't in the search results,say you don't know
        instead of guessing.
    """
)

In [78]:
def ask_hr_assistant(question: str) -> str:
    response = hr_assistant.invoke({
        "messages":[
            {
                "role":"user",
                "content": question
            }
        ]
        }
    )
    answer = response["messages"][-1].content

    print("Answer:",answer)
    print()
    return answer

In [79]:
ask_hr_assistant("How many sick leaves employees get?")

Answer: Employees are entitled to **10 paid sick days per year**. (The leave policy states that “Sick leave is separate from annual leave, and employees get 10 paid sick days per year.”)



'Employees are entitled to **10 paid sick days per year**. (The leave policy states that “Sick leave is separate from annual leave, and employees get\u202f10\u202fpaid sick days per year.”)'